![Week 1 Day 5](../assets/banners/w1d5.png)

# Week 1 Day 5 - 🧱 Project Day: Asha's Exam Pack → JSON

It's project day! 🎉

This week you've counted tokens and dollars, made a model answer in Pydantic shapes and fix its own mistakes, sent it images of PDF pages, and handed it tools. Yesterday's recap promised that you now hold every ingredient for GradeWise's first real milestone. Today we cook.

Asha has handed us the full CS201 midterm pack: a **question paper**, a **marking scheme** and a **marking methodology**, plus her **model answers**. By the end of this notebook, GradeWise turns all of it into **one validated `ExamPaper`**: a JSON array of questions, each carrying its own mark points and marking method. Every later week grades against this file, so we'll also check it, field by field, against the answer key.

First, though, let's step back and look at the whole machine you're building.

![Your 6-week roadmap](../assets/roadmap/week1.png)

### 🎬 Videos for this lab
| # | Lecture |
|:-:|---|
| 1 | 🧱 Day 5 - GradeWise Architecture Walkthrough: What We'll Build |
| 2 | 🧱 Day 5 - Designing the Exam Schema, Then Extracting the Paper to JSON |
| 3 | Day 5 - Week 1 Wrap-Up: Summary, Roadmap and Quiz |

### 🎯 By the end of this lab you will
1. 🗺️ Know every stage of GradeWise, and which week builds it
2. 📐 Understand *why* the exam schema looks the way it does
3. 🧱 Run the real extraction on the exam pack and save `output/cs_midterm.json`
4. 🔍 Compare the result with `expected_exam.json`, field by field

In [ ]:
# Imports and environment first, all in one place

import json
import os
import subprocess
import sys

import tiktoken
from dotenv import load_dotenv

from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.exam import exam_pack, exam_summary, extract_exam, parse_pdf, print_exam_summary
from gradewise.exam.extract import SYSTEM_PROMPT, build_messages
from gradewise.schemas import ExamPaper, MarkingCriterion, Question

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: The GradeWise architecture

Here's the whole system. Each box is something you'll build, and the week that builds it:

![Diagram](../assets/diagrams/123709508f2d.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  subgraph W2["Week 2 · Knowledge Base"]
    A[Course PDFs] --> B[pymupdf4llm loader] --> C[Chunker + metadata] --> D[Embeddings] --> E[(Chroma)]
  end
  subgraph W1W3["Weeks 1 & 3 · Exam Understanding"]
    F[Question paper] --> H[Extractor<br/>structured output]
    G[Marking scheme + methodology] --> H
    H --> I{Validate} -- retry --> H
    I --> J[[Admin approval]] --> K[(Exam JSON)]
  end
  subgraph W4["Week 4 · Transcription"]
    L[Answer PDF] --> M[Pages → images] --> N[Vision LLM, verbatim] --> O[Answer → question mapper]
  end
  subgraph W5["Week 5 · Grading & Reports"]
    Q[One grader per question] --> R[Grader agent + tools] --> S[Guardrails] --> T[[Human review]] --> U[Report cards]
  end
  E -. evidence .-> R
  K --> Q
  O --> Q
```

</details>


**Today you build the box called *Extractor*** (week 3 adds *Validate → retry* and *Admin approval*). Notice the arrow from **Exam JSON** into grading: the exam JSON is the contract every grader reads. If a mark is wrong here, every student's grade is wrong later. That's why today ends with a careful comparison.

| Week | GradeWise learns to... | Lives in |
|:-:|---|---|
| 1 | understand an exam: pack → `ExamPaper` JSON | `gradewise/exam/`, `gradewise/schemas/exam.py` |
| 2 | know the course: PDFs → chunks → Chroma, with hybrid search | `gradewise/ingestion/`, `gradewise/rag/` |
| 3 | check its own work and ask Asha to approve (LangGraph) | `gradewise/exam/graph.py` |
| 4 | read handwriting, faithfully, with confidence flags | `gradewise/transcription/` |
| 5 | grade with rubric + evidence, then write report cards | `gradewise/grading/`, `gradewise/reports/` |
| 6 | become an app: FastAPI + Streamlit + Docker | `gradewise/api/`, `gradewise/ui/` |

The package you'll grow is already in the repo, fully working. Let's take a quick tour of what's there today:

In [ ]:
from gradewise.config import REPO_ROOT

package = REPO_ROOT / "gradewise"
for folder in sorted(p for p in package.iterdir() if p.is_dir() and not p.name.startswith("__")):
    modules = sorted(f.stem for f in folder.glob("*.py") if f.stem != "__init__")
    print(f"📦 gradewise/{folder.name + '/':<16} {', '.join(modules)}")

## Part 2: Designing the exam schema

Good schemas start from the **consumer**, not the documents. So ask: *what does a grader need, for ONE answer, to mark it fairly?*

| A grader needs... | So the schema has... | Why |
|---|---|---|
| to know which answer goes with which question | a stable `id` per **markable unit** (`Q7a`, `Q7b`) | students label answers "Q7(b)"; week 4 maps them by ID |
| the question, on its own | `text` with the parent's shared stem copied in | Q7(b) is meaningless without the IP address in Q7's stem |
| the mark points | `criteria`: a list of `MarkingCriterion` with `marks` | week 5 grades and explains **per criterion** |
| how to mark *this* type of answer | `methodology` on **every** question | each grader call is self-contained: no need to re-read three PDFs |
| what to look up in the course notes | `reference_topics` | week 4's RAG retrieves evidence with these |
| (optionally) a full-marks answer | `model_answer` | a reference, never a requirement |

And one design choice from Day 3: Pydantic checks the **shape**, while `ExamPaper.problems()` checks the **arithmetic** and returns a list instead of raising, so it can be sent back to the model.

Open `gradewise/schemas/exam.py` in your editor now: it's short. Here's its outline, read straight from the classes:

In [ ]:
for model in [ExamPaper, Question, MarkingCriterion]:
    print(f"\n{model.__name__}: {model.__doc__.strip()}")
    for name, field in model.model_fields.items():
        kind = getattr(field.annotation, "__name__", str(field.annotation)).replace("typing.", "")
        required = "" if field.is_required() else " (optional)"
        print(f"   {name:<20} {kind[:40]:<40}{required}")

### The answer key: `expected_exam.json`

Asha checked one extraction by hand and saved it as `data/exams/cs_midterm/expected_exam.json`. It's our **answer key**. Let's load it through the schema and look at it with GradeWise's summary table:

In [ ]:
expected = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
print_exam_summary(expected)
print("problems():", expected.problems() or "none 🎉")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The paper's instructions say <i>"Answer ALL ten questions"</i>, yet the table has <b>15</b> rows. Why? Which design rule from the table above causes it? And what would go wrong in week 5 if we had stored Question 7 as one 5-mark entry instead of <code>Q7a</code> and <code>Q7b</code>?</span>
        </td>
    </tr>
</table>

In [ ]:
# 💡 One complete entry: notice the shared stem ("A server on a company network...") copied into Q7b
q7b = next(q for q in expected.questions if q.id == "Q7b")
print(q7b.model_dump_json(indent=2))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><code>expected_exam.json</code> is for <b>checking</b> the extractor, never for <b>feeding</b> it. If you paste the answer key into the prompt, of course the output matches, and you've learned nothing about how GradeWise will do on next term's exam. Keeping test data out of the model's input is the first rule of evaluation, and we'll lean on it hard in weeks 2 and 5.</span>
        </td>
    </tr>
</table>

## Part 3: Reading the exam pack

`exam_pack()` finds the files for an exam ID, and complains clearly if one is missing. Its keys deliberately match the arguments of `extract_exam`, as you'll see in Part 4.

In [ ]:
pack = exam_pack("cs_midterm")
for key, path in pack.items():
    print(f"{key:<15} → {path.relative_to(DATA_DIR)}")

`parse_pdf` (in `gradewise/exam/parse.py`) is the Day 4 idea, grown up: `pymupdf4llm` turns each page into Markdown, and then **repeated headers and footers** are stripped, so the model doesn't read "CONFIDENTIAL: for markers only" eight times.

In [ ]:
docs = {key: parse_pdf(path) for key, path in pack.items() if path.suffix == ".pdf"}

for key, doc in docs.items():
    print(f"{key:<15} {doc.page_count} pages, {len(doc.text):>6,} chars. Stripped: {doc.stripped_lines}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">A true story from building this course: the first version of <code>parse_pdf</code> let a footer slip into Question 5. The question paper has only <b>two</b> pages, and its page-1 footer ("Page 1 of 2 (turn over)") differs from page 2's, so it never <i>repeated</i>. Predict: how could a parser catch a footer that appears only once?</span>
        </td>
    </tr>
</table>

In [ ]:
import pymupdf4llm

from gradewise.exam.parse import KNOWN_RUNNING_LINES, strip_repeated_lines

raw_pages = [
    c["text"] for c in pymupdf4llm.to_markdown(str(pack["question_paper"]), page_chunks=True, show_progress=False)
]

# 💡 Repetition only (how the parser first worked) vs repetition + known running lines (how it works now)
old_way, _ = strip_repeated_lines(raw_pages, known_lines=())
new_way, stripped = strip_repeated_lines(raw_pages)

for label, pages in [("repetition only", old_way), ("+ known_lines", new_way)]:
    text = "\n\n".join(pages)
    start = text.index("**Question 5**")
    print(f"── {label} ──\n{text[start : start + 160]}\n")

print("Known running-line patterns:", KNOWN_RUNNING_LINES)

**The fix:** besides "lines that repeat at the page edges", `strip_repeated_lines` now also drops edge lines that match a short list of patterns we *know* Northbridge prints (`KNOWN_RUNNING_LINES`, overridable with `known_lines=`), even if they appear only once. A guard makes sure a line like "Question 5" is never treated as a footer. The lesson: **cleaning is never perfect on the first try**. We only found this leak because we looked at the text, which is exactly why Part 5 checks the output instead of trusting the input.

Now the prompt. `build_messages` puts the system prompt (a numbered checklist, every rule written after a model got it wrong) and all four documents under clear headings into one conversation. Let's measure it before we spend anything:

In [ ]:
messages = build_messages(
    docs["question_paper"].text,
    docs["marking_scheme"].text,
    docs["methodology"].text,
    model_answers=pack["model_answers"].read_text(encoding="utf-8"),
)

encoding = tiktoken.get_encoding("o200k_base")
for m in messages:
    print(f"{m.type:<7} {len(encoding.encode(m.content)):>6,} tokens")

print("\n" + SYSTEM_PROMPT[:900] + "…")

## Part 4: Extract the exam 🧱

The moment of truth. `extract_exam` reads the files, builds the messages, calls `chat_model(temperature=0).with_structured_output(ExamPaper)`, forces the `exam_id` (we choose the storage key, not the model), and warns about any `problems()`.

Because `exam_pack()`'s keys match `extract_exam`'s arguments, the call is one line. It takes a little while: the model is writing about 11,000 tokens of JSON.

In [ ]:
paper = extract_exam(**pack)  # 💡 question_paper=..., marking_scheme=..., methodology=..., model_answers=...

print_exam_summary(paper)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / "cs_midterm.json"
out_path.write_text(json.dumps(paper.model_dump(), indent=2, ensure_ascii=False), encoding="utf-8")
print(f"💾 Saved {out_path}")
print(json.dumps(exam_summary(paper), indent=2)[:600] + "…")  # 💡 the compact view week 3 shows Asha

### The same milestone, as a script

The notebook is for learning; the milestone also exists as a plain script. Open `week1/project/extract_exam.py` and its `README.md`. It's the same idea in about 100 readable lines: read with `pymupdf4llm`, one `with_structured_output(ExamPaper)` call, check `problems()`, save. In a terminal, from the repo root, run:

```bash
uv run week1/project/extract_exam.py
```

Or run it from here (the output appears once it finishes):

In [ ]:
script = REPO_ROOT / "week1" / "project" / "extract_exam.py"
run = subprocess.run([sys.executable, str(script)], cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8")
print(run.stdout[-3000:] or run.stderr[-3000:])
print("exit code:", run.returncode)

## Part 5: Did we get it right? A field-by-field diff

`problems()` tells us the arithmetic is consistent. It can't tell us the extraction is **correct**: a model could merge two criteria into one and still make the numbers add up. For that we compare with the answer key.

Our comparison rules:
- **IDs, numbers, types and marks must match exactly.** A wrong mark is a wrong grade.
- **Free text only has to be similar enough.** "Copy the text" can still differ by a space or a curly quote, so we score similarity from 0 to 1 with Python's `difflib`. Methodology is *composed* from several sections, so we're more lenient there.

In [ ]:
# 🧪 TRY: change a threshold and see how the diff below changes

from difflib import SequenceMatcher

from rich.table import Table

from gradewise.console import console


def similarity(a: str | None, b: str | None) -> float:
    """0.0 (nothing alike) → 1.0 (identical), ignoring case and extra whitespace."""
    a, b = " ".join((a or "").lower().split()), " ".join((b or "").lower().split())
    return 1.0 if a == b else SequenceMatcher(None, a, b).ratio()


def compare_exams(expected: ExamPaper, actual: ExamPaper, text_threshold: float = 0.9) -> list[dict]:
    """Field-by-field differences between two ExamPapers. An empty list means they match.

    💡 Numbers, IDs and types must match EXACTLY. Free text only has to be similar enough,
       because "copy the text" can still differ by a space or a curly quote.
    """
    rows: list[dict] = []

    def report(where: str, field: str, want, got, kind: str) -> None:
        rows.append({"where": where, "field": field, "expected": want, "actual": got, "kind": kind})

    def text(where: str, field: str, want: str | None, got: str | None, threshold: float = text_threshold) -> None:
        score = similarity(want, got)
        if score < threshold:
            report(where, field, (want or "")[:60], (got or "")[:60], f"text {score:.0%} similar")

    for field in ["title", "subject", "total_marks"]:
        if getattr(expected, field) != getattr(actual, field):
            report("paper", field, getattr(expected, field), getattr(actual, field), "value")

    want_qs = {q.id: q for q in expected.questions}
    got_qs = {q.id: q for q in actual.questions}
    for qid in want_qs.keys() - got_qs.keys():
        report(qid, "question", qid, None, "missing")
    for qid in got_qs.keys() - want_qs.keys():
        report(qid, "question", None, qid, "extra")

    for qid in [q for q in want_qs if q in got_qs]:  # 💡 keep the paper's order
        want, got = want_qs[qid], got_qs[qid]
        for field in ["number", "question_type", "max_marks"]:
            if getattr(want, field) != getattr(got, field):
                report(qid, field, getattr(want, field), getattr(got, field), "value")
        text(qid, "text", want.text, got.text)
        text(qid, "methodology", want.methodology, got.methodology, threshold=0.5)  # 💡 composed, so looser
        if (want.model_answer is None) != (got.model_answer is None):
            report(qid, "model_answer", want.model_answer is not None, got.model_answer is not None, "present?")
        elif want.model_answer:
            text(qid, "model_answer", want.model_answer, got.model_answer)

        want_cs = {c.id: c for c in want.criteria}
        got_cs = {c.id: c for c in got.criteria}
        for cid in want_cs.keys() - got_cs.keys():
            report(cid, "criterion", f"worth {want_cs[cid].marks:g}", None, "missing")
        for cid in got_cs.keys() - want_cs.keys():
            report(cid, "criterion", None, f"worth {got_cs[cid].marks:g}", "extra")
        for cid in [c for c in want_cs if c in got_cs]:
            if want_cs[cid].marks != got_cs[cid].marks:
                report(cid, "marks", want_cs[cid].marks, got_cs[cid].marks, "value")
            text(cid, "description", want_cs[cid].description, got_cs[cid].description, threshold=0.7)
    return rows


def show_diff(rows: list[dict]) -> None:
    """A readable table of compare_exams() rows, or a green tick if there are none."""
    if not rows:
        print("✅ No differences: the two exams match field by field.")
        return
    table = Table(header_style="accent")
    for column in ["where", "field", "expected", "actual", "kind"]:
        table.add_column(column, overflow="fold")
    for row in rows:
        table.add_row(*(str(row[c]) for c in ["where", "field", "expected", "actual", "kind"]))
    console.print(table)
    print(f"{len(rows)} difference(s). The serious ones: missing / extra / value rows.")

First, a sanity check that runs offline: an exam compared with itself must show **no** differences. (Always test your test!)

In [ ]:
show_diff(compare_exams(expected, expected))

Now let's make sure it actually *catches* things. We copy the answer key and introduce the classic extraction mistakes by hand: a merged sub-part, a misread mark, a dropped criterion, a paraphrased question and a messy ID.

In [ ]:
broken = expected.model_dump()
qs = {q["id"]: q for q in broken["questions"]}

qs["Q5"]["max_marks"] = 4  # a misread mark...
qs["Q5"]["criteria"] = qs["Q5"]["criteria"][:4]  # ...and a dropped criterion, so problems() stays happy!
qs["Q3"]["text"] = "Explain deadlock and list two conditions for it."  # paraphrased, not copied
qs["Q10a"]["id"] = "Q10(a)"  # an ID with brackets
broken["questions"] = [q for q in broken["questions"] if q["id"] != "Q7b"]  # Q7b merged away
broken["total_marks"] = 46  # 50 - 1 (Q5) - 3 (Q7b): the totals still add up

broken = ExamPaper.model_validate(broken)
print("problems():", broken.problems() or "none: the arithmetic checks can't see these mistakes!")
show_diff(compare_exams(expected, broken))

Every planted mistake shows up, including the ones `problems()` missed. Now the real test: your extraction against the answer key.

In [ ]:
rows = compare_exams(expected, paper)
show_diff(rows)

**How to read your diff:**
- 🟥 **missing / extra / value** rows are real errors. A different `question_type` is worth a look; a different `max_marks` or a missing criterion must be fixed before grading.
- 🟨 **text** rows on `methodology` or `description` are usually harmless rewording. Read a couple to be sure.
- Run it again: is the diff the same each time? (That's today's exercise.)

Don't be surprised if v1 isn't perfect. One call, with no checking, is exactly the weakness **week 3** fixes: `extract_with_retry` sends `problems()` back to the model, and a LangGraph workflow pauses for Asha to approve or edit the exam before it's stored.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Today's milestone lives in <code>gradewise/exam/</code>: <code>parse.py</code> (PDF → Markdown, footers stripped), <code>extract.py</code> (<code>build_messages</code>, <code>extract_once</code>, <code>extract_exam</code>, and week 3's <code>extract_with_retry</code>) and <code>summary.py</code> (the table you've been looking at). The standalone version is <code>week1/project/extract_exam.py</code>. In week 3, <code>gradewise/exam/graph.py</code> wraps it in a LangGraph workflow with retries and admin approval.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">You just turned a stack of exam documents into a typed, validated, testable data structure in a single call, for a few cents. That pattern (messy documents → schema → validation → diff against a gold set) is one of the most valuable things you can sell as an AI engineer: invoices, contracts, insurance claims, lab reports and job specs all look like Asha's exam pack once you squint.</span>
        </td>
    </tr>
</table>

## Part 6: Week 1 wrap-up

What a week! Here's what you built, day by day:

| Day | You learned | GradeWise gained |
|:-:|---|---|
| 1 | Setup, `.env`, your first LLM call | a working environment and `gradewise.models` |
| 2 | Tokens, context windows, temperature, cost; invoke / stream / batch | a cost model: an exam extraction costs cents |
| 3 | Prompt templates, Pydantic, `with_structured_output`, self-correction | the extraction pattern and `problems()` |
| 4 | Images as content blocks; `@tool`, `bind_tools`, the tool loop | the ingredients for transcription (wk 4) and the grader (wk 5) |
| 5 | Schema design, extraction, field-by-field evaluation | **milestone 1: `output/cs_midterm.json`** 🧱 |

![Week 1 complete](../assets/roadmap/week1.png)

📝 **Quiz 1** is waiting for you on Udemy at the end of the section (10 questions, about 10 minutes; a practice copy with explanations is in `quizzes/week_1.yaml`). It mixes concepts, "spot the bug" code and GradeWise design questions.

🏋️ **The week 1 exercise**, `week1/week1_EXERCISE.ipynb`, asks you to add **multiple-choice questions** to the exam schema, without editing the package. It's the best way to make this week's ideas your own.

## Recap, and where we are heading

Today you:
- 🗺️ toured GradeWise's architecture and saw where each week's work lands
- 📐 designed the exam schema from the grader's point of view: markable units, stable IDs, self-contained questions
- 🧱 extracted Asha's whole exam pack into a validated `ExamPaper` and saved `output/cs_midterm.json`
- 🔍 built a field-by-field diff, tested it on planted mistakes, and checked your extraction against the answer key

GradeWise now understands the *exam*. But to grade fairly it must understand the *course*: what Asha actually taught about deadlocks, paging and DNS. **Next week** we build GradeWise's knowledge base: loading the CS201 notes, chunking them, embedding them into Chroma, and retrieving the right evidence with hybrid search. Welcome to RAG!

⏭ **Next:** `week1/week1_EXERCISE.ipynb`, then `week2/day1.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>How consistent is v1?</b> Run <code>extract_exam(**pack)</code> twice and use <code>compare_exams</code> on the <i>two runs</i> (not against the key). Which fields change between runs, and which never do? Then extract once more <i>without</i> the model answers (hint: <code>{k: v for k, v in pack.items() if k != "model_answers"}</code>) and diff again: what happens to <code>model_answer</code>? Bonus: try your <code>SMART_MODEL</code> via <code>extract_exam(..., llm=make_chat_model(settings.smart_model))</code>. Is it worth the price? Solution in <code>week1/solutions/day5_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">Thank you!</h2>
            <span style="color:#7c5cff;">That's week 1 done, and GradeWise's first milestone shipped. Thank you for sticking with the setup and the theory: it pays off from here on. If you're enjoying the course, a quick rating on Udemy really helps other learners find it. See you in week 2! 🚀</span>
        </td>
    </tr>
</table>